# Load Dataset from HF

This will gather the dataset created using the dataset and load it into memory.


In [ ]:
# lib import
import os
from datasets import load_dataset, get_dataset_config_names

# setup
data = {}
source_ds_cache_dir = os.path.join(os.getcwd(), "data", "wikipos-test")
configs = get_dataset_config_names("whatphiliptrains/wikipos-test")

for config in configs:
    data[config] = load_dataset(
        "whatphiliptrains/wikipos-test", config, cache_dir=source_ds_cache_dir
    )

test = data[configs[-1]]

# verification (optional)
print(f"Dataset length: {len(test['train'])}")
print(test["train"][0])
print(test["train"][-1])

# Trustworthiness and Continuity

Using `scikit-learn` the trustworthiness (and continuity) between the dense embedding and the reduced position vector is calculated


In [ ]:
import numpy as np
from sklearn.manifold import trustworthiness
from sklearn.neighbors import NearestNeighbors

n = 15
results = {}

def calculate_continuity(X, X_embedded, n_neighbors=5):
    """
    Calculate continuity metric for dimensionality reduction.
    
    Continuity measures whether points that are close in the low-dimensional 
    embedding are also close in the high-dimensional space.
    """
    # Ensure X is a numpy array
    if not isinstance(X, np.ndarray):
        X = np.array(X)
    if not isinstance(X_embedded, np.ndarray):
        X_embedded = np.array(X_embedded)
    
    # Find nearest neighbors in low-dimensional space
    nbrs_embedded = NearestNeighbors(n_neighbors=n_neighbors + 1).fit(X_embedded)
    _, indices_embedded = nbrs_embedded.kneighbors(X_embedded)
    
    # Find nearest neighbors in high-dimensional space
    nbrs_original = NearestNeighbors(n_neighbors=n_neighbors + 1).fit(X)
    _, indices_original = nbrs_original.kneighbors(X)
    
    continuity_sum = 0
    n_samples = X.shape[0]
    
    for i in range(n_samples):
        # Get k nearest neighbors in embedded space (excluding self)
        embedded_neighbors = set(indices_embedded[i][1:n_neighbors + 1])
        
        # Get k nearest neighbors in original space (excluding self)
        original_neighbors = set(indices_original[i][1:n_neighbors + 1])
        
        # Count how many embedded neighbors are also original neighbors
        intersection = len(embedded_neighbors.intersection(original_neighbors))
        continuity_sum += intersection / n_neighbors
    
    return continuity_sum / n_samples

for config in configs:
    current = data[config]["train"]
    pos = np.column_stack([current["x"], current["y"]])
    embeddings = np.array(current["embeddings"])

    trust = trustworthiness(X=embeddings, X_embedded=pos, n_neighbors=n)
    continuity = calculate_continuity(X=embeddings, X_embedded=pos, n_neighbors=n)

    results[config] = {"trustworthiness": trust, "continuity": continuity}
    print(f"[{config}] : trust: {trust:.4f}, continuity: {continuity:.4f}")

print(f"\nResults stored in 'results' dictionary with {len(results)} configurations.")

[all_MiniLM_L6_v2_tsne] : trust: 0.9600, continuity: 0.2921
[all_MiniLM_L6_v2_umap] : trust: 0.9311, continuity: 0.2228
[all_MiniLM_L6_v2_umap] : trust: 0.9311, continuity: 0.2228
[all_MiniLM_L6_v2_umap_pca] : trust: 0.9241, continuity: 0.1954
[all_MiniLM_L6_v2_umap_pca] : trust: 0.9241, continuity: 0.1954
[all_mpnet_base_v2_tsne] : trust: 0.9642, continuity: 0.3135
[all_mpnet_base_v2_tsne] : trust: 0.9642, continuity: 0.3135
[all_mpnet_base_v2_umap] : trust: 0.9354, continuity: 0.2447
[all_mpnet_base_v2_umap] : trust: 0.9354, continuity: 0.2447
[all_mpnet_base_v2_umap_pca] : trust: 0.9279, continuity: 0.2074
[all_mpnet_base_v2_umap_pca] : trust: 0.9279, continuity: 0.2074
[nomic_embed_text_v1_5_tsne] : trust: 0.9768, continuity: 0.3449
[nomic_embed_text_v1_5_tsne] : trust: 0.9768, continuity: 0.3449
[nomic_embed_text_v1_5_umap] : trust: 0.9458, continuity: 0.2678
[nomic_embed_text_v1_5_umap] : trust: 0.9458, continuity: 0.2678
[nomic_embed_text_v1_5_umap_pca] : trust: 0.9610, continui